In [2]:
import pandas as pd

X_train = pd.read_csv("creditCardFraud/data/processed/X_train_scaled.csv")
X_test = pd.read_csv("creditCardFraud/data/processed/X_test_scaled.csv")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

X_train: (226980, 30)
X_test : (56746, 30)


In [3]:
from scipy.stats import ks_2samp

drift_results = {}

for feature in X_train.columns:
    statistic, p_value = ks_2samp(
        X_train[feature],
        X_test[feature]
    )

    drift_results[feature] = {
        "ks_statistic": statistic,
        "p_value": p_value,
        "drift_detected": p_value < 0.05
    }

drift_df = pd.DataFrame(drift_results).T

drift_df.sort_values(
    "ks_statistic",
    ascending=False
).head(10)

,ks_statistic,p_value,drift_detected
V21,0.007397,0.01384,True
V28,0.007179,0.018478,True
V23,0.006815,0.029373,True
V14,0.006365,0.050307,False
Time,0.006008,0.075145,False
V19,0.005422,0.138068,False
Amount,0.005328,0.151313,False
V5,0.004917,0.221741,False
V22,0.004853,0.234589,False
V17,0.004725,0.262075,False


In [ ]:
import joblib
import numpy as np

model = joblib.load("creditCardFraud/models/final_xgb_model.joblib")
threshold = joblib.load("creditCardFraud/models/fraud_threshold.joblib")

test_probabilities = model.predict_proba(X_test)[:, 1]

predicted_fraud_rate = (
    test_probabilities >= threshold
).mean()

print("Mean fraud probability:", test_probabilities.mean())
print("Predicted fraud rate:", predicted_fraud_rate)